# Tratamento dos Dados (Silver)

## Objetivos
- Padronizar nomes das colunas para snake_case
- Tratar valores nulos e inconsistentes
- Converter tipos de dados apropriadamente
- Extrair colunas úteis para análises gráficas
- Preparar dados para visualizações (pizza, regressão linear, boxplot)

## Estrutura dos Dados Originais
O dataset contém informações sobre escolas brasileiras do INEP com 19 colunas principais:
- Informações geográficas (UF, Município, Localização, Latitude, Longitude)
- Características administrativas (Dependência, Categoria, Porte)
- Informações educacionais (Etapas de Ensino, Modalidades)
- Status operacional (Restrição de Atendimento, Regulamentação)

In [ ]:
# === IMPORTS ===
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, split, size, coalesce, lit, regexp_replace, lower, regexp_extract
from pyspark.sql.types import IntegerType, DoubleType, StringType
import pyspark.sql.functions as F
import os
from pathlib import Path


In [ ]:
# === INICIALIZAÇÃO DO SPARK ===
spark = (SparkSession.builder
    .appName("inep_schools_analysis")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate())

print("✓ Spark Session inicializada")


In [ ]:
# === CONFIGURAÇÃO DO CAMINHO DOS DADOS BRONZE ===
bronze_env_path = os.getenv("BRONZE_DATA_PATH", "../Data Layer/raw/dados_brutos.csv")
bronze_csv_path = Path(bronze_env_path)
if not bronze_csv_path.is_absolute():
    bronze_csv_path = (Path.cwd() / bronze_csv_path).resolve()

print("=== UTILIZANDO ARQUIVO BRONZE ===")
print(bronze_csv_path)


In [ ]:
# === CARREGAMENTO DOS DADOS BRONZE ===
df = spark.read.csv(str(bronze_csv_path), header=True, sep=",")

print("=== ESTRUTURA INICIAL DOS DADOS ===")
print(f"Total de registros: {df.count():,}")
print(f"Número de colunas: {len(df.columns)}")
df.printSchema()


In [ ]:
# === ANÁLISE DE VALORES NULOS ===
print("=== ANÁLISE DE VALORES NULOS ===")
null_counts = df.select([F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c) for c in df.columns]).collect()[0]
for col_name in df.columns:
    null_count = null_counts[col_name]
    if null_count > 0:
        print(f"{col_name}: {null_count:,} valores nulos/vazios")


In [ ]:
df_clean = df.withColumnRenamed("Restrição de Atendimento", "rst_atn") \
             .withColumnRenamed("Escola", "nom_esc") \
             .withColumnRenamed("Código INEP", "cod_inep") \
             .withColumnRenamed("UF", "uf") \
             .withColumnRenamed("Município", "mun") \
             .withColumnRenamed("Localização", "lca") \
             .withColumnRenamed("Localidade Diferenciada", "loc_diferenciada") \
             .withColumnRenamed("Categoria Administrativa", "categoria_adm") \
             .withColumnRenamed("Endereço", "endereco") \
             .withColumnRenamed("Telefone", "telefone") \
             .withColumnRenamed("Dependência Administrativa", "dpd_adm") \
             .withColumnRenamed("Categoria Escola Privada", "categoria_esc_privada") \
             .withColumnRenamed("Conveniada Poder Público", "conveniada_poder_publico") \
             .withColumnRenamed("Regulamentação pelo Conselho de Educação", "regulamentacao_conselho") \
             .withColumnRenamed("Porte da Escola", "prt_esc") \
             .withColumnRenamed("Etapas e Modalidade de Ensino Oferecidas", "etp_mod") \
             .withColumnRenamed("Outras Ofertas Educacionais", "outras_ofertas") \
             .withColumnRenamed("Latitude", "lat") \
             .withColumnRenamed("Longitude", "lon")

print("=== COLUNAS RENOMEADAS PARA MNEMÔNICOS ===")
df_clean.printSchema()


In [ ]:
# === CONVERSÃO DE COORDENADAS PARA DOUBLE ===
df_clean = df_clean.withColumn("lat", 
    when(col("lat").rlike("^-?\\d+\\.?\\d*$"), col("lat").cast(DoubleType()))
    .otherwise(None)
).withColumn("lon", 
    when(col("lon").rlike("^-?\\d+\\.?\\d*$"), col("lon").cast(DoubleType()))
    .otherwise(None)
)

print("✓ Coordenadas convertidas para DoubleType")


In [ ]:
# === LIMPEZA DE STRINGS (REMOVER ESPAÇOS EXTRAS) ===
string_columns = ["rst_atn", "nom_esc", "uf", "mun", 
                 "lca", "categoria_adm", "dpd_adm",
                 "prt_esc", "etp_mod"]

for col_name in string_columns:
    df_clean = df_clean.withColumn(col_name, trim(col(col_name)))

print("✓ Strings limpas (espaços extras removidos)")


In [ ]:
# === TRATAMENTO DE VALORES "NÃO INFORMADO" ===
df_clean = df_clean.withColumn("categoria_esc_privada",
    when(col("categoria_esc_privada") == "Não Informado", None)
    .otherwise(col("categoria_esc_privada"))
)

print("✓ Valores 'Não Informado' convertidos para NULL")


In [ ]:
# === FILTRAGEM: APENAS ESCOLAS COM COORDENADAS VÁLIDAS ===
df_clean = df_clean.filter(
    col("lat").isNotNull() & 
    col("lon").isNotNull() &
    (col("lat") != 0) & 
    (col("lon") != 0)
)

print(f"=== DADOS APÓS LIMPEZA ===")
print(f"Registros válidos: {df_clean.count():,}")
print(f"Registros removidos: {df.count() - df_clean.count():,}")


In [ ]:
# === COLUNA DERIVADA: NÚMERO DE ETAPAS/MODALIDADES ===
df_clean = df_clean.withColumn("etp_list", 
    split(regexp_replace(col("etp_mod"), r",\\s*", ","), ",")
).withColumn("qtd_etp", 
    coalesce(size(col("etp_list")), lit(0))
)

print("✓ Coluna 'qtd_etp' criada (NULLs substituídos por 0)")


In [ ]:
# === COLUNA DERIVADA: PORTE NUMÉRICO ===
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def map_porte_to_numeric(porte):
    if porte is None:
        return None
    porte_lower = porte.lower().strip()
    if "pequeno" in porte_lower or "até 50" in porte_lower:
        return 1
    elif "médio" in porte_lower or "51" in porte_lower or "201" in porte_lower or "501" in porte_lower:
        return 2
    elif "grande" in porte_lower or "1000" in porte_lower:
        return 3
    else:
        return None

map_porte_udf = udf(map_porte_to_numeric, IntegerType())
df_clean = df_clean.withColumn("prt_num", map_porte_udf(col("prt_esc")))

print("✓ Coluna 'prt_num' criada (1=Pequeno, 2=Médio, 3=Grande)")


In [ ]:
# === COLUNA DERIVADA: FLAG ESCOLAS RURAIS ===
df_clean = df_clean.withColumn("is_rur", 
    when(col("lca") == "Rural", 1).otherwise(0)
)

print("✓ Coluna 'is_rur' criada (1=Rural, 0=Urbana)")


In [ ]:
# === COLUNA DERIVADA: FLAG ESCOLAS PÚBLICAS ===
df_clean = df_clean.withColumn("is_pub", 
    when(col("dpd_adm").isin(["Estadual", "Municipal", "Federal"]), 1)
    .otherwise(0)
)

print("✓ Coluna 'is_pub' criada (1=Pública, 0=Privada)")


In [ ]:
# === COLUNA DERIVADA: REGIÃO DO BRASIL ===
def get_regiao(uf):
    regioes = {
        'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
        'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 
        'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
        'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
        'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
        'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
    }
    return regioes.get(uf, 'Outro')

get_regiao_udf = udf(get_regiao, StringType())
df_clean = df_clean.withColumn("reg", get_regiao_udf(col("uf")))

print("✓ Coluna 'reg' criada")


In [ ]:
# === RESUMO DAS COLUNAS DERIVADAS ===
print("=== COLUNAS DERIVADAS CRIADAS ===")
print("Colunas adicionadas:")
print("- num_etapas: número de etapas/modalidades oferecidas")
print("- porte_numerico: porte da escola (1=Pequeno, 2=Médio, 3=Grande)")
print("- is_rural: flag para escolas rurais (1=Rural, 0=Urbana)")
print("- is_publica: flag para escolas públicas (1=Pública, 0=Privada)")
print("- regiao: região do Brasil baseada na UF")


In [ ]:
# === DEFINIÇÃO DAS COLUNAS PARA ANÁLISE ===
colunas_analise = [
    "cod_inep", "nom_esc", "uf", "mun", "reg",
    "lca", "is_rur", "dpd_adm", "is_pub",
    "prt_esc", "prt_num", "etp_mod", "qtd_etp",
    "lat", "lon", "rst_atn"
]

print("✓ Colunas para análise definidas")


In [ ]:
# === SELEÇÃO FINAL DAS COLUNAS ===
df_final = df_clean.select(*colunas_analise)

print("=== DADOS FINAIS PARA ANÁLISE ===")
print(f"Total de registros: {df_final.count():,}")
print(f"Colunas selecionadas: {len(colunas_analise)}")
print()
print("Colunas incluídas:")
for col in colunas_analise:
    print(f"- {col}")


In [ ]:
# === CONFIGURAÇÃO DE CONEXÃO COM POSTGRESQL ===
postgres_host = os.getenv("POSTGRES_HOST", os.getenv("DB_HOST", "localhost"))
postgres_port = os.getenv("POSTGRES_PORT", os.getenv("DB_PORT", "5432"))
postgres_db = os.getenv("POSTGRES_DB", "inep_db")
postgres_user = os.getenv("POSTGRES_USER", "inep")
postgres_password = os.getenv("POSTGRES_PASSWORD", "inep")
postgres_table = os.getenv("POSTGRES_TABLE", "esc")
postgres_schema = os.getenv("POSTGRES_SCHEMA", "silver")

jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}?currentSchema={postgres_schema}"
jdbc_properties = {
    "user": postgres_user,
    "password": postgres_password,
    "driver": "org.postgresql.Driver"
}

print("=== CONFIGURAÇÃO POSTGRESQL ===")
print(f"Host: {postgres_host}")
print(f"Port: {postgres_port}")
print(f"Database: {postgres_db}")
print(f"Table: {postgres_table}")


In [ ]:
# === LIMPEZA DA TABELA POSTGRESQL ===
print("=== LIMPANDO TABELA POSTGRESQL ===")
print(f"Tabela: {postgres_table}")

(df_final.limit(0).write
    .mode("overwrite")
    .option("truncate", "true")
    .jdbc(url=jdbc_url, table=postgres_table, properties=jdbc_properties))

print("✓ Tabela limpa (truncate executado)")


In [ ]:
# === COLETA DOS DADOS PARA INSERÇÃO ===
print("=== COLETANDO DADOS DO DATAFRAME ===")
total_records = df_final.count()
rows = df_final.collect()

print(f"✓ {total_records:,} registros coletados para inserção")


In [ ]:
# === VERIFICAÇÃO E IMPORTAÇÃO DO PSYCOPG2 ===
try:
    import psycopg2
    from psycopg2.extras import execute_values
    USE_PSYCOPG2 = True
    print("✓ psycopg2 encontrado - usando inserção otimizada")
except ImportError:
    print("⚠️  psycopg2 não encontrado. Usando Spark JDBC linha por linha (mais lento).")
    USE_PSYCOPG2 = False


In [ ]:
# === INSERÇÃO NO POSTGRESQL (MÉTODO: PSYCOPG2) ===
if USE_PSYCOPG2:
    print("=== INSERINDO DADOS NO POSTGRESQL (LINHA POR LINHA) ===")
    print(f"Destino: {postgres_table} @ {postgres_host}:{postgres_port}/{postgres_db}")
    print(f"Total de registros: {total_records:,}")
    
    # Conectar ao PostgreSQL
    conn = psycopg2.connect(
        host=postgres_host,
        port=postgres_port,
        database=postgres_db,
        user=postgres_user,
        password=postgres_password
    )
    cur = conn.cursor()
    
    # Preparar query de inserção
    columns = ", ".join(colunas_analise)
    placeholders = ", ".join(["%s"] * len(colunas_analise))
    insert_query = f"INSERT INTO {postgres_schema}.{postgres_table} ({columns}) VALUES ({placeholders})"
    
    # Inserir linha por linha
    inserted = 0
    for idx, row in enumerate(rows, 1):
        values = []
        for col in colunas_analise:
            val = row[col]
            # Tratar valores NULL para colunas NOT NULL
            if val is None:
                if col == "qtd_etp":
                    val = 0
                else:
                    val = None
            values.append(val)
        
        cur.execute(insert_query, values)
        inserted += 1
        
        # Commit a cada 100 registros para melhor performance
        if inserted % 100 == 0:
            conn.commit()
            print(f"  Linha {inserted:,}/{total_records:,} inserida...", end='\r')
    
    # Commit final
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"\n✓ Dados gravados com sucesso no banco PostgreSQL!")
    print(f"  Total: {inserted:,} registros inseridos linha por linha")


In [ ]:
# === INSERÇÃO NO POSTGRESQL (MÉTODO: SPARK JDBC - FALLBACK) ===
if not USE_PSYCOPG2:
    print("=== INSERINDO DADOS NO POSTGRESQL (LINHA POR LINHA - SPARK JDBC) ===")
    print(f"Destino: {postgres_table} @ {postgres_host}:{postgres_port}/{postgres_db}")
    print(f"Total de registros: {total_records:,}")
    
    inserted = 0
    for idx, row in enumerate(rows, 1):
        # Criar lista de valores na ordem das colunas
        row_data = [tuple([row[col] for col in colunas_analise])]
        
        # Criar DataFrame com uma única linha usando o schema do df_final
        single_row_df = spark.createDataFrame(row_data, schema=df_final.schema)
        
        # Inserir linha no banco
        (single_row_df.write
            .mode("append")
            .jdbc(url=jdbc_url, table=postgres_table, properties=jdbc_properties))
        
        inserted += 1
        
        # Mostrar progresso a cada 100 registros
        if inserted % 100 == 0 or inserted == total_records:
            print(f"  Linha {inserted:,}/{total_records:,} inserida...", end='\r')
    
    print(f"\n✓ Dados gravados com sucesso no banco PostgreSQL!")
    print(f"  Total: {inserted:,} registros inseridos linha por linha")
